# PostGIS Spatial SQL — 10 Interview Flashcards
## Python + PostgreSQL/PostGIS + Pandas/GeoPandas

**Project:** Pull PostGIS spatial query results directly into Pandas/GeoPandas DataFrames for analysis.

This notebook is a study and reference guide based on the project tables discussed:
- `public.indian_districts_7755` — district boundaries (`gid`, `statename`, `distname`, `geom`)
- `public.river_buffer_5km` — river buffer geometries (`geom`)
- `public.river_buffer_district_analysis` — district attribute/area summary

**Important:** Run the SQL only against a database where you have permission. The connection cell prompts for your password rather than storing it in the notebook. Some examples use `ST_DWithin` and `ST_Buffer`; verify geometry units and CRS before interpreting distances as metres. EPSG:7755 is a metre-based projected CRS, but validate suitability for the geographic extent and your analysis requirements.


## 1. Connect Python to PostgreSQL

Install packages in your project virtual environment if needed:

```powershell
.\.venv\Scripts\python.exe -m pip install sqlalchemy psycopg2-binary pandas geopandas
```

The following cell creates a SQLAlchemy connection. It does not print or save your password.


In [2]:
print("Hello! My Python notebook is working.")

Hello! My Python notebook is working.


In [ ]:

from getpass import getpass
from sqlalchemy import create_engine, URL, text
import pandas as pd

password = getpass("Enter PostgreSQL password: ")

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password=password,
    host="localhost",
    port=5432,
    database="agriculture_db",
)

engine = create_engine(
    connection_url,
    connect_args={"connect_timeout": 5}
)

with engine.connect() as connection:
    print("Database connection successful!")
    print(connection.execute(text("SELECT version();")).scalar())
    print(
        "PostGIS version:",
        connection.execute(
            text("SELECT PostGIS_Full_Version();")
        ).scalar()
    )



In [1]:
print("Notebook kernel is responding.")

Notebook kernel is responding.


In [ ]:

from getpass import getpass
from sqlalchemy import create_engine, URL, text
import pandas as pd

password = getpass("PostgreSQL password: ")

engine = create_engine(
    URL.create(
        "postgresql+psycopg2",
        username="postgres",
        password=password,
        host="127.0.0.1",
        port=5432,
        database="agriculture_db",
    ),
    connect_args={"connect_timeout": 5},
)

print("Testing database connection...")

with engine.connect() as connection:
    result = connection.execute(text("SELECT 1")).scalar()
    print("Connection successful:", result)

print("Ready to run SQL queries.")


## 2. Ten spatial SQL interview flashcards

Each card contains an interview question, a SQL example, and the concept to explain. Examples using placeholder tables are explicitly labelled; adjust names and IDs before execution.

### 1. ST_Intersects — Which districts touch river buffers?

**Query**

In [ ]:
SELECT d.gid, d.statename, d.distname
FROM public.indian_districts_7755 AS d
WHERE EXISTS (
    SELECT 1
    FROM public.river_buffer_5km AS b
    WHERE ST_Intersects(d.geom, b.geom)
)
ORDER BY d.statename, d.distname;

**Explanation / interview answer**

`ST_Intersects(A, B)` is true when two geometries share any point. `EXISTS` returns each district once even if many buffer features intersect it. This project query previously returned 689 districts.

### 2. ST_Intersection — What portion of a district overlaps the buffer?

**Query**

In [ ]:
WITH buffer_union AS (
    SELECT ST_UnaryUnion(ST_Collect(geom)) AS geom
    FROM public.river_buffer_5km
)
SELECT
    d.gid,
    d.statename,
    d.distname,
    ST_Area(ST_Intersection(d.geom, b.geom)) / 1000000.0
        AS overlap_area_sq_km
FROM public.indian_districts_7755 AS d
CROSS JOIN buffer_union AS b
WHERE ST_Intersects(d.geom, b.geom);

**Explanation / interview answer**

`ST_Intersection` creates the shared geometry. `ST_Area` measures that overlap; dividing by 1,000,000 converts square metres to square kilometres for a metre-based CRS. Dissolving buffers first prevents overlapping buffer polygons from being counted repeatedly.

### 3. ST_Area — How large is each district?

**Query**

In [ ]:
SELECT
    gid,
    statename,
    distname,
    ST_Area(geom) / 1000000.0 AS area_sq_km
FROM public.indian_districts_7755
ORDER BY area_sq_km DESC
LIMIT 10;

**Explanation / interview answer**

`ST_Area` returns the planar area for a geometry. Because EPSG:7755 uses metres, the area is in square metres before conversion. For national-scale results, confirm the CRS is suitable for the full extent and required accuracy.

### 4. ST_Buffer — How do we create a 5 km buffer?

**Query**

In [ ]:
-- Example only: assumes geom is in a suitable projected CRS whose units are metres.
SELECT
    gid,
    ST_Buffer(geom, 5000) AS buffer_5km
FROM public.some_river_layer;

**Explanation / interview answer**

`ST_Buffer(geom, 5000)` creates a buffer 5,000 coordinate units from the input geometry. Use this distance only when the CRS units are metres and the CRS is appropriate for the area being measured. Replace `some_river_layer` with the actual source table if you run it.

### 5. ST_DWithin — Which river features are within 5 km?

**Query**

In [ ]:
-- Example only: use an appropriate metre-based projected CRS.
SELECT d.gid, d.statename, d.distname
FROM public.indian_districts_7755 AS d
WHERE EXISTS (
    SELECT 1
    FROM public.river_buffer_5km AS b
    WHERE ST_DWithin(d.geom, b.geom, 5000)
);

**Explanation / interview answer**

`ST_DWithin(A, B, distance)` checks whether geometries are within a specified distance. It can use spatial indexes. This example compares districts with buffer polygons, so it is not the same as measuring distance to the original river centerline.

### 6. ST_Contains — Which points are inside a district?

**Query**

In [ ]:
-- Example: replace public.sample_points with your point table.
SELECT
    p.id AS point_id,
    d.gid AS district_id,
    d.statename,
    d.distname
FROM public.sample_points AS p
JOIN public.indian_districts_7755 AS d
  ON ST_Contains(d.geom, p.geom);

**Explanation / interview answer**

`ST_Contains(A, B)` tests whether geometry A contains geometry B. A point exactly on a polygon boundary is not considered contained; use `ST_Covers` if boundary inclusion is required. The sample table name and column names are placeholders.

### 7. ST_Distance — What is the distance between two geometries?

**Query**

In [ ]:
-- Example: compare a district to one selected river feature.
SELECT
    d.gid,
    d.distname,
    ST_Distance(d.geom, r.geom) AS distance_in_crs_units
FROM public.indian_districts_7755 AS d
JOIN public.some_river_layer AS r
  ON r.id = 1
WHERE d.gid = 1;

**Explanation / interview answer**

`ST_Distance` returns the shortest planar distance between geometries. In EPSG:7755 the output unit is metres, subject to CRS suitability. The table names and IDs are illustrative and must be adapted to your actual river source table.

### 8. ST_Transform — How do we convert a geometry to WGS 84?

**Query**

In [ ]:
SELECT
    gid,
    statename,
    distname,
    ST_Transform(geom, 4326) AS geom_wgs84
FROM public.indian_districts_7755
LIMIT 5;

**Explanation / interview answer**

`ST_Transform` changes coordinates from the geometry's source CRS to the target SRID. It transforms coordinates rather than merely changing the SRID label. EPSG:4326 is commonly used for GeoJSON longitude/latitude coordinates.

### 9. ST_Union / ST_UnaryUnion — How do we dissolve overlapping buffers?

**Query**

In [ ]:
SELECT ST_UnaryUnion(ST_Collect(geom)) AS dissolved_buffer_geom
FROM public.river_buffer_5km;

**Explanation / interview answer**

`ST_Collect` groups geometries without dissolving their boundaries. `ST_UnaryUnion` then merges overlapping parts into a combined geometry. This is useful before calculating the total covered area so overlapping buffers do not double-count area.

### 10. ST_SRID and GeometryType — How do we inspect spatial data?

**Query**

In [ ]:
SELECT
    ST_SRID(geom) AS srid,
    GeometryType(geom) AS geometry_type,
    COUNT(*) AS feature_count
FROM public.indian_districts_7755
GROUP BY ST_SRID(geom), GeometryType(geom);

**Explanation / interview answer**

`ST_SRID` reports the spatial reference identifier stored on the geometry; `GeometryType` reports types such as POLYGON or MULTIPOLYGON. Matching SRIDs are necessary for many spatial operations but do not alone prove the CRS is appropriate for every measurement.

## 3. Load a spatial SQL result into Pandas

This query returns attributes only, so use `pandas.read_sql_query` rather than `geopandas.read_postgis`.


In [ ]:
query = """
SELECT
    d.gid AS district_id,
    d.statename,
    d.distname
FROM public.indian_districts_7755 AS d
WHERE EXISTS (
    SELECT 1
    FROM public.river_buffer_5km AS b
    WHERE ST_Intersects(d.geom, b.geom)
)
ORDER BY d.statename, d.distname;
"""

df = pd.read_sql_query(query, engine)
print("Shape:", df.shape)
display(df.head())
print(df["statename"].value_counts())


## 4. Load geometries into GeoPandas

Use `geopandas.read_postgis` when the SQL result includes a geometry column. The `geom_col` argument must match the selected geometry column name.


In [ ]:
import geopandas as gpd

spatial_query = """
SELECT
    gid AS district_id,
    statename,
    distname,
    geom
FROM public.indian_districts_7755
LIMIT 10;
"""

gdf = gpd.read_postgis(spatial_query, engine, geom_col="geom")
print("Shape:", gdf.shape)
print("CRS:", gdf.crs)
print(gdf.geom_type.value_counts())
display(gdf.head())


## 5. Coverage validation reminder

Do not assume every 100% coverage value is correct just because the query ran. Validate:
1. District and buffer geometry SRIDs and coordinate units.
2. The source CRS's suitability for area measurement across the whole study area.
3. That the buffer union is used to avoid double-counting overlapping buffers.
4. That `buffer_covered_area_sq_km <= district_area_sq_km` within reasonable numerical tolerance.
5. That the reported coverage is `covered area / district area * 100`.

The previously observed 85.47% mean should be treated as provisional until these checks pass.


In [ ]:
# Optional validation for a previously exported CSV
from pathlib import Path

coverage_file = Path("output/river_buffer_coverage_by_district.csv")
if coverage_file.exists():
    coverage_df = pd.read_csv(coverage_file)
    print("Rows:", len(coverage_df))
    print("Coverage summary:")
    print(coverage_df["coverage_percentage"].describe())
    print(
        "Rows where covered area exceeds district area:",
        (
            coverage_df["buffer_covered_area_sq_km"]
            > coverage_df["district_area_sq_km"] + 0.01
        ).sum()
    )
else:
    print(f"Not found: {coverage_file}. Run the coverage script first.")


In [ ]:
# Close the database connection when finished
engine.dispose()
print("Database engine disposed.")
